# pytorch_nnfs
Let's learn some object oriented programming!

**2026-08-01**
Created this document. I plan to implement an object oriented version of my older nnfs code.

In [2]:
import pandas
import torch
import matplotlib

# initialising torch accelerators
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using {device} device")

g = torch.Generator(device=device)

Using mps device


In [3]:
# data
data = torch.from_numpy(pandas.read_csv('../../data/train.csv').to_numpy())

def one_hot(data, outputs):
    return torch.eye(outputs)[Y_train]

# data preparation
X_train = data[:32000,1:].float()/255
Y_train = data[:32000,0]
Y_train_one_hot = one_hot(Y_train, 10)

X_val = data[32000:36000,1:].float()/255
Y_val = data[32000:36000,0]

X_test = data[36000:42000,1:].float()/255
Y_test = data[36000:42000,0]

# move data to the gpu
X_train = X_train.to(device)
Y_train = Y_train.to(device)
Y_train_one_hot = Y_train_one_hot.to(device)

# Main
To debug:
- Layers.backward()
- Rewrite Layers class

In [4]:
# initialisation functions
class init:
    class weight_init:
        @staticmethod
        def input_initialisation(neurons, device=device):
            return torch.ones(neurons, 1, device=device)

        def output_initialisation(neurons, device=device):
            return torch.ones(neurons, 1, device=device)

        @staticmethod
        def He_initialisation(neurons, num_weights, device=device):
            return torch.randn(neurons, num_weights, device=device) * torch.sqrt(2 / torch.tensor(num_weights))

    class bias_init:
        @staticmethod
        def zero_initialisation(neurons, device):
            return torch.zeros(neurons, 1, device=device)
    

init_dict = {
    'input': init.weight_init.input_initialisation,
    'output': init.weight_init.output_initialisation,
    'he': init.weight_init.He_initialisation,
}

bias_dict = {
    'zero': init.bias_init.zero_initialisation,
}

# activation functions
class activation_functions:
    class ReLU:
        @staticmethod
        def forward(Z):
            return torch.clamp(Z, min=0)

        @staticmethod
        def backward(Z): # i.e. derivative
            return (Z > 0).float()

    class Softmax:
        @staticmethod
        def forward(Z):
            max_Z = torch.max(Z, dim=0, keepdim=True).values
            exp_Z = torch.exp(Z - max_Z) # Z - max_Z to make sure we don't have an exploding number. maybe that's why it's called exp?
            sum_exp_Z = torch.sum(exp_Z, dim=0, keepdim=True)
            return exp_Z / sum_exp_Z

        @staticmethod
        def backward(Z):
            pass
            # add back propagation later

# Layer class
class Layer:
    def __init__(self, type, neurons=None, num_weights=None, weight_init=None, bias_init=None, device=device):
        self.type=type

        # neurons: the number of neurons
        # num_weights: the number of neurons in the last layer.

        # layers that require initialisation
        if type == 'input':
            self.weights = init_dict[weight_init](neurons, device)

        if type == 'output':
            self.weights = init_dict[weight_init](neurons, device)

        if type == 'linear':
            self.weights = init_dict[weight_init](neurons, num_weights, device)
            self.bias = bias_dict[bias_init](neurons, device)

        # unified layers & activation functions
        if type == 'linear_softmax':
            self.weights = init_dict[weight_init](neurons, num_weights, device)
            self.bias = bias_dict[bias_init](neurons, device)

    def __repr__(self):
        return str(self.type) + '\n'

    def forward(self, data):

        # layers
        if self.type == 'input':
            self.inputs = data
            self.activations = data.reshape(784,-1)
            return self.activations
        if self.type == 'output':
            self.inputs = data
            self.pre_activations = data.reshape(10,-1)
            self.activations = torch.argmax(self.pre_activations, axis=0)
            return self.pre_activations
        if self.type == 'linear':
            self.inputs = data
            self.pre_activations = self.weights @ data + self.bias
            return self.pre_activations

        # special cases because back propagation is a pain
        if self.type == 'linear_softmax':
            self.inputs = data
            self.pre_activations = self.weights @ data + self.bias
            self.activations = activation_functions.Softmax.forward(self.pre_activations)
            return self.activations

        # activation functions
        if self.type == 'relu':
            self.activations = activation_functions.ReLU.forward(data)
            return self.activations

    def backward(self, loss_fn='CrossEntropyLoss', delta=None, Y_one_hot=None, batch_size=1): # implement batch_size later

        # layers
        if self.type == 'input':
            return None
        if self.type == 'output': # add this later
            return self.inputs
        if self.type == 'linear':
            delta = self.pre_activations
            self.dW = delta @ self.inputs.T
            self.db = delta
        if self.type == 'linear_softmax': # add this later
            delta = self.pre_activations - Y_one_hot
            self.dW = delta @ self.inputs.T
            self.db = delta

            return delta

        # activation functions
        if self.type == 'relu':
            self.delta = delta
            self.activation_function_deriv = activation_functions.ReLU.backward(data)

"""
  dZ1 = (W2.T @ dZ2) * relu.deriv(Z1)
  dW1 = dZ1 @ X.T
  db1 = torch.sum(dZ1, dim=1, keepdim=True)
"""
            
        

class MLP:
    def __init__(self, layer_config):
        self.layers = []
        for i in range(len(layer_config)):
            self.layers.append(Layer(**layer_config[i]))

    def __repr__(self):
        return str(self.layers)

    def forward(self, data): # debug this
        outputs = []
        output = self.layers[0].forward(data)
        print(output.shape)
        outputs.append(output)
        for i in range(1, len(self.layers)):
            output = self.layers[i].forward(outputs[i-1])
            outputs.append(output)
            print(output.shape)

        return outputs

    #def backward(self, data)

layer_config = [
    {'type': 'input',         #0
     'neurons': 784,
     'weight_init': 'input',
     'device': device},
    {'type': 'linear',        #1
     'neurons': 128,
     'num_weights': 784,
     'weight_init': 'he',
     'bias_init': 'zero',
     'device': device},
    {'type': 'relu'},         #2
    {'type': 'linear',        #3
     'neurons': 32,
     'num_weights': 128,
     'weight_init': 'he',
     'bias_init': 'zero',
     'device': device},
    {'type': 'relu'},         #4
    {'type': 'linear_softmax',#5
     'neurons': 10,
     'num_weights': 32,
     'weight_init': 'he',
     'bias_init': 'zero',
     'device': device},
    {'type': 'output',        #6
     'neurons': 10,
     'weight_init': 'output',
     'device': device}
]

MLP_object = MLP(layer_config)
outputs = MLP_object.forward(X_train[1])
outputs[6].sum(dim=0)

torch.Size([784, 1])
torch.Size([128, 1])
torch.Size([128, 1])
torch.Size([32, 1])
torch.Size([32, 1])
torch.Size([10, 1])
torch.Size([10, 1])


tensor([1.0000], device='mps:0')